> **Public release copy.** Set `<DATA_DIR>` in the CONFIG cell to the folder holding the large input files (see README / DATA availability). Internal validation cells have been removed. The small curation TSVs are included in this folder.


## CONFIG -- edit paths here

In [ ]:
library(tidyverse)
library(data.table)

# -- CONFIG -- edit paths here -----------------------------------------------
workingpath <- getwd()
outpath0    <- "<DATA_DIR>"

# Inputs
reduced_matrix_file <- "salmon_counts_fishassociated_reduced.tsv.gz"  # from 02a (workingpath)
curated_file        <- "curated_contigs.tsv"                              # from 01 (workingpath)
metadata_file       <- "all_zf_dates_devstage_tissue_tech_curated.tsv"    # <- UPDATE to newest metadata

# Outputs
out_allmeta    <- "curated_virus_counts_byrun_withallmetadata.tsv"
out_selectmeta <- "curated_virus_counts_byrun_withselectmetadata.tsv"     # -> SUPPLEMENTAL

# Metadata columns kept in the supplemental file, in this order (updated June 2026 metadata).
# bioproject is also kept and relocated to the front below.
select_metadata_cols <- c(
  "seqdetective.n_mates",
  "seqdetective.mapping_rate.mate1",   "seqdetective.mapping_rate.mate2",
  "seqdetective.nofeature_rate.mate1", "seqdetective.nofeature_rate.mate2",
  "seqdetective.sparsity.mate1",       "seqdetective.sparsity.mate2",
  "seqdetective.pos_strand_rate.mate1","seqdetective.pos_strand_rate.mate2",
  "seqdetective.readlen.mate1",        "seqdetective.readlen.mate2",
  "seqdetective.judgement.mate1",      "seqdetective.judgement.mate2",
  "seqdetective.judgement.reason",
  "platform_family", "instrument_generation", "read_bias", "selection_class",
  "prep_kit", "sc_or_bulk", "tech_class", "technology", "tech_variant",
  "submission.bioprojectsource.country", "earliest_date",
  "devstage_curation", "devstage_curation_coarse",
  "tissue_curation", "tissue_curation_coarse"
)

# Sprivivirus NTtarget fallback (the one column-name special case)
sprivi_curated_name    <- "Spring viraemia of carp virus"
sprivi_column_signature <- "CLUSTER1_Sprivivirus_cyprinus"  # 3rd '|' field of the 2 NTtarget cols
# ----------------------------------------------------------------------------

## 1. Load reduced matrix + curated map

In [ ]:
Sys.setenv("VROOM_CONNECTION_SIZE" = 50 * 1024 * 1024)

df <- read_tsv(file.path(workingpath, reduced_matrix_file), show_col_types = FALSE)
cat("Reduced matrix:", nrow(df), "rows x", ncol(df), "cols\n")

curated <- read_tsv(file.path(workingpath, curated_file), show_col_types = FALSE)
# contig name -> curated virus + Category
contig_map <- curated %>%
  distinct(contig_withLCA_withcluster, clusterLCA_curated, Category)
cat("Curated contigs:", nrow(contig_map),
    "| curated viruses:", n_distinct(contig_map$clusterLCA_curated), "\n")

## 2. Map each matrix column -> curated virus (+ Sprivivirus fallback)

In [ ]:
run_col   <- names(df)[1]
data_cols <- names(df)[-1]

col_map <- tibble(column = data_cols) %>%
  left_join(contig_map, by = c("column" = "contig_withLCA_withcluster")) %>%
  # Sprivivirus NTtarget fallback: the 2 reference-genome columns are not in curated_contigs.tsv
  mutate(clusterLCA_curated = if_else(
           is.na(clusterLCA_curated) & str_detect(column, fixed(sprivi_column_signature)),
           sprivi_curated_name, clusterLCA_curated))

n_unmapped <- sum(is.na(col_map$clusterLCA_curated))
cat("Matrix columns:", length(data_cols),
    "| mapped to a curated virus:", sum(!is.na(col_map$clusterLCA_curated)),
    "| UNMAPPED:", n_unmapped, "\n")
if (n_unmapped > 0) {
  cat("WARNING -- unmapped columns (should be 0; investigate):\n")
  print(head(col_map$column[is.na(col_map$clusterLCA_curated)], 20))
}

# Confirm Sprivivirus got its columns
cat("Sprivivirus columns mapped:",
    sum(col_map$clusterLCA_curated == sprivi_curated_name, na.rm = TRUE), "(expect 2)\n")

## 3. Collapse columns sharing a curated name (rowSums)

In [ ]:
dt <- as.data.table(df)
curated_names <- col_map$clusterLCA_curated     # parallel to data_cols
setnames(dt, old = data_cols, new = curated_names)

result_list <- list(dt[[run_col]]); names(result_list) <- run_col
for (cl in unique(curated_names)) {
  idx <- which(names(dt) == cl)
  result_list[[cl]] <- if (length(idx) == 1) dt[[idx]] else rowSums(dt[, ..idx], na.rm = TRUE)
}
counts <- as_tibble(result_list)

# Round to integer counts
counts <- counts %>% mutate(across(-all_of(run_col), ~as.integer(round(.x, 0))))
cat("Collapsed:", nrow(counts), "runs x", ncol(counts) - 1, "curated viruses\n")

## 4. Order virus columns by Category, then descending total + recapitulate category prefixes

Orders by Category (Novel -> Known -> Endogenous -> Insufficient) then descending total within
each. Then **recapitulates the old column-name prefixes** so the output matches the historical
file: Endogenous viruses get an `endogenous_or_nonfish ` prefix and Insufficient viruses get an
`insufficient_evidence ` prefix; Novel/Known keep their clean names.

In [ ]:
cat_order <- c("Novel", "Known", "Endogenous or not fish-associated", "Insufficient Evidence")

virus_cat <- contig_map %>% distinct(clusterLCA_curated, Category) %>%
  add_row(clusterLCA_curated = sprivi_curated_name,
          Category = contig_map$Category[match(sprivi_curated_name, contig_map$clusterLCA_curated)]) %>%
  distinct(clusterLCA_curated, .keep_all = TRUE)

totals <- counts %>% select(-all_of(run_col)) %>%
  summarise(across(everything(), ~sum(.x, na.rm = TRUE))) %>%
  pivot_longer(everything(), names_to = "clusterLCA_curated", values_to = "total") %>%
  left_join(virus_cat, by = "clusterLCA_curated") %>%
  mutate(Category = factor(Category, levels = cat_order)) %>%
  arrange(Category, desc(total))

# --- recapitulate the historical category prefixes on the virus column names ---
prefix_for <- function(cat) dplyr::case_when(
  cat == "Endogenous or not fish-associated" ~ "endogenous_or_nonfish ",
  cat == "Insufficient Evidence"             ~ "insufficient_evidence ",
  TRUE                                        ~ "")                 # Novel / Known unchanged
totals <- totals %>% mutate(display_name = paste0(prefix_for(as.character(Category)), clusterLCA_curated))

clean_order  <- totals$clusterLCA_curated      # current (clean) column names in counts
display_order<- totals$display_name            # desired output names (with prefixes)

counts <- counts %>% select(all_of(run_col), all_of(clean_order))
# rename clean -> prefixed
names(counts)[match(clean_order, names(counts))] <- display_order
ordered_cols <- display_order                  # downstream uses the prefixed names

cat("Ordered", length(ordered_cols), "virus columns; applied category prefixes\n")
print(totals %>% group_by(Category) %>% summarise(n = n(), .groups = "drop"))

## 5. Attach metadata + row-gap check

Strip the `_salmon` suffix from run names, then left-join metadata. An earlier version flagged a
row-count discrepancy (matrix ~73,955 rows vs anndata 74,275, ~350 NA) -- this cell reports it
explicitly so it's visible rather than silent.

In [ ]:
metadata <- read_tsv(file.path(workingpath, metadata_file), show_col_types = FALSE)
cat("Metadata:", nrow(metadata), "rows;", ncol(metadata), "cols\n")

counts <- counts %>% mutate(!!run_col := str_remove(.data[[run_col]], "_salmon$"))

# --- row-gap check ---
n_matrix   <- nrow(counts)
n_in_meta  <- sum(counts[[run_col]] %in% metadata$run.accession)
cat("Row-gap check:\n")
cat("  matrix runs:", n_matrix, "\n")
cat("  runs matched in metadata:", n_in_meta, "\n")
cat("  matrix runs NOT in metadata (-> NA metadata):", n_matrix - n_in_meta, "\n")

df_all <- counts %>%
  left_join(metadata, by = setNames("run.accession", run_col))
# Coerce ONLY the virus count columns to integer (NOT metadata numerics -- dates / large IDs
# can exceed R's integer range and would silently become NA, which a blanket coercion would risk).
df_all <- df_all %>% mutate(across(all_of(ordered_cols), as.integer))

# The reduced matrix's run column is "run_name"; the supplemental table (and old S1) use
# "run.accession". Rename it here so both outputs match S1.
if (run_col != "run.accession") {
  df_all <- df_all %>% rename(run.accession = all_of(run_col))
  run_col <- "run.accession"
}
cat("With all metadata:", nrow(df_all), "x", ncol(df_all), "\n")

## 6. Write outputs (counts + supplemental select-metadata)

In [ ]:
# -> (full) all metadata columns
write_tsv(df_all, file.path(workingpath, out_allmeta))
cat("Wrote", out_allmeta, ":", nrow(df_all), "x", ncol(df_all), "\n")

# -> SUPPLEMENTAL DATA FILE: virus count columns + selected metadata
missing_meta <- setdiff(select_metadata_cols, names(df_all))
if (length(missing_meta) > 0) {
  cat("WARNING -- these select_metadata_cols are not in the metadata (check column names):\n")
  print(missing_meta)
}
n_virus_cols <- length(ordered_cols)   # virus count columns
# Column order (matches old Supplemental S1): run.accession, then bioproject, then the 53 virus
# count columns (Category-ordered, prefixed), then the selected metadata columns in order.
lead_cols <- intersect(c(run_col, "bioproject"), names(df_all))   # run.accession first, then bioproject
if (!"bioproject" %in% names(df_all))
  cat("NOTE: 'bioproject' not in joined data -- leading with", run_col, "only.\n")
df_select <- df_all %>%
  select(all_of(lead_cols), all_of(ordered_cols), any_of(select_metadata_cols))

write_tsv(df_select, file.path(workingpath, out_selectmeta))
cat("Wrote", out_selectmeta, "(SUPPLEMENTAL):", nrow(df_select), "x", ncol(df_select), "\n")